# Deployment Parameter Configuration Updater

This notebook dynamically updates configuration files by replacing placeholders with actual Fabric resource IDs.

## Workflow Options

### Option 1: Configuration Update Only (Original)
1. **Discover Resources** - Fetch workspace ID, lakehouses, and notebooks
2. **Resolve IDs** - Map placeholder names to actual resource IDs
3. **Update Configuration** - Replace placeholders in template with actual values
4. **Validate & Save** - Validate JSON and save updated configuration

### Option 2: Configuration Update + Copy (New Feature)
1. **Discover Resources** - Fetch workspace ID, lakehouses, and notebooks
2. **Resolve IDs** - Map placeholder names to actual resource IDs
3. **Update Configuration** - Replace placeholders in template with actual values
4. **Validate & Save** - Validate JSON and save updated configuration
5. **Copy Configurations** - Copy configuration folders to admin lakehouse
   - `system-configurations` → `{admin_lakehouse}/Files/system-configurations`
   - `_internal` → `{admin_lakehouse}/Files/HealthDataManager/DMHConfiguration/_internal`

## Prerequisites

- Access to Microsoft Fabric workspace
- Template configuration file with placeholders (%%placeholder%%)
- Sufficient permissions to read resources and write configuration files
- **For Option 2 only**: Admin lakehouse created (from lakehouses_and_tables_deployer) and configuration folders in distribution path

## Instructions

1. Update configuration parameters in the `common_deployment_config` notebook:
   - `WORKSPACE_NAME` - Fabric workspace name (used as the OneLake container; **case-sensitive**)
   - `ARTIFACT_LAKEHOUSE_NAME` - Lakehouse containing the distribution files
   - Any other deployment parameters as needed
2. In this notebook, verify the derived paths (based on `WORKSPACE_NAME`) in the configuration cell.
3. Choose execution option in the final cell (Option 1: config only, Option 2: config + copy).


---

## Import Libraries

In [ ]:
%run common_deployment_config

---

## Configuration Parameters

Configure the template and output file locations below. Paths are derived from the base distribution path, which itself is built from `WORKSPACE_NAME` and `ARTIFACT_LAKEHOUSE_NAME` in `common_deployment_config`.

### File Configuration
- `WORKSPACE_NAME` - Fabric workspace name used as the OneLake container (set in `common_deployment_config`)
- `BASE_DIST_PATH` - Base distribution path computed from `WORKSPACE_NAME` and `ARTIFACT_LAKEHOUSE_NAME`
- `CONFIG_SOURCE_FOLDER` - Folder path containing the template configuration file
- `CONFIG_TARGET_FOLDER` - Folder path where the updated configuration will be written (can differ from source)
- `TEMPLATE_FILENAME` - Name of the template configuration file
- `OUTPUT_FILENAME` - Name of the output configuration file
- `TEMPLATE_FULL_PATH` - Full path to the template (constructed from `CONFIG_SOURCE_FOLDER` and `TEMPLATE_FILENAME`)
- `OUTPUT_FULL_PATH` - Full path to the output (constructed from `CONFIG_TARGET_FOLDER` and `OUTPUT_FILENAME`)

### Lakehouse Placeholder Mapping
- `LAKEHOUSE_PLACEHOLDER_CONFIG` - Maps template placeholders to manifest keys and fallback patterns
  - Uses `LakehouseHydrationManifest.json` as the single source of truth
  - Each placeholder specifies:
    - `manifest_key` - Key in the manifest (e.g., "admin", "bronze", "poa-gold")
    - `fallback_patterns` - Alternative name patterns to try if prefixed name not found
  - Resolver tries patterns in order: prefixed manifest key → prefixed fallbacks → unprefixed patterns

### Naming Prefixes
- `COMPANY_PREFIX` & `TECHNICAL_PREFIX` - Naming prefixes (configured in `common_deployment_config`)

> **Note:** The OneLake endpoint, prefixes, and lakehouse manifest are loaded from `common_deployment_config`. Source and target folders are kept as separate variables for flexibility, even when they point to the same location. Lakehouse resolution is now configuration-driven via `LAKEHOUSE_PLACEHOLDER_CONFIG` instead of hardcoded patterns.

In [ ]:
# ============================================================================
# NOTEBOOK-SPECIFIC CONFIGURATION
# ============================================================================
# (Common config loaded from common_deployment_config)

# Configuration folder paths (separate variables for source and target for flexibility)
CONFIG_SOURCE_FOLDER: str = (
    f"{BASE_DIST_PATH}/healthcare-configuration/{ARTIFACT_VERSION}/system-configurations"
)
CONFIG_TARGET_FOLDER: str = (
    f"{BASE_DIST_PATH}/healthcare-configuration/{ARTIFACT_VERSION}/system-configurations"
)

# Template and output file names
TEMPLATE_FILENAME: str = "deploymentParametersConfiguration.template.json"
OUTPUT_FILENAME: str = "deploymentParametersConfiguration.json"

# Construct full paths
TEMPLATE_FULL_PATH: str = f"{CONFIG_SOURCE_FOLDER}/{TEMPLATE_FILENAME}"
OUTPUT_FULL_PATH: str = f"{CONFIG_TARGET_FOLDER}/{OUTPUT_FILENAME}"

# ============================================================================
# LAKEHOUSE PLACEHOLDER MAPPING CONFIGURATION
# ============================================================================
# Maps template placeholder names to manifest keys and fallback patterns.
# This is the single source of truth for lakehouse name resolution.
#
# Structure:
#   "template_placeholder_name": {
#       "manifest_key": "key in LakehouseHydrationManifest.json",
#       "fallback_patterns": [list of fallback name patterns to try]
#   }
#
# The resolver will:
#   1. Try build_artifact_name(manifest_key) - prefixed name from manifest
#   2. Try each fallback_pattern in order
#   3. Report missing if none match

LAKEHOUSE_PLACEHOLDER_CONFIG = {
    "administration_lakehouse_id": {
        "manifest_key": "admin",
        "fallback_patterns": ["admin"],  # Unprefixed fallback
    },
    "bronze_lakehouse_id": {"manifest_key": "bronze", "fallback_patterns": ["bronze"]},
    "silver_lakehouse_id": {"manifest_key": "silver", "fallback_patterns": ["silver"]},
    "omop_lakehouse_id": {
        "manifest_key": "omop",
        "fallback_patterns": ["gold_omop", "omop"],  # Try alternative names
    },
    "customer_insights_lakehouse_id": {
        "manifest_key": None,  # Not in manifest - external/optional lakehouse
        "fallback_patterns": [
            "customerinsights",
            "customer_insights",
        ],  # Try both naming styles
    },
    "patient_outreach_lakehouse_id": {
        "manifest_key": "poa-gold",  # Note: hyphenated in manifest
        "fallback_patterns": [
            "poa_gold",
            "patient_outreach",
        ],  # Try underscore and descriptive
    },
    "care_management_analytics_lakehouse_id": {
        "manifest_key": "cma-gold",  # Note: hyphenated in manifest
        "fallback_patterns": ["gold_cma", "care_management_analytics", "cma_gold"],
    },
}

print("✓ Config updater configuration:")
print(f"  Workspace: {WORKSPACE_NAME}")
print(f"  Source Folder: {CONFIG_SOURCE_FOLDER}")
print(f"  Target Folder: {CONFIG_TARGET_FOLDER}")
print(f"  Template File: {TEMPLATE_FILENAME}")
print(f"  Output File: {OUTPUT_FILENAME}")
print(f"  Template Full Path: {TEMPLATE_FULL_PATH}")
print(f"  Output Full Path: {OUTPUT_FULL_PATH}")
print(f"  Lakehouse Placeholders Configured: {len(LAKEHOUSE_PLACEHOLDER_CONFIG)}")

---

## Define Functions

All functions are defined below for resource discovery and configuration updates.

In [ ]:
def get_workspace_id() -> str:
    """
    Get current workspace ID from common config (already auto-detected).

    Returns:
        Workspace ID string
    """
    workspace_id = WORKSPACE_ID
    print(f"  ℹ️  Workspace ID: {workspace_id}")
    return workspace_id


def fetch_lakehouses(workspace_id: str) -> Dict[str, str]:
    """
    Fetch all lakehouses in workspace and create name->ID mapping.

    Args:
        workspace_id: Fabric workspace ID

    Returns:
        Dictionary mapping lakehouse names to IDs
    """
    print("  ℹ️  Fetching lakehouses...")
    fabric_client = FabricRestClient()
    url = f"/v1/workspaces/{workspace_id}/lakehouses"

    try:
        response = fabric_client.get(url)
        response.raise_for_status()
        lakehouses = response.json().get("value", [])
        lakehouse_map = {
            lh.get("displayName"): lh.get("id")
            for lh in lakehouses
            if lh.get("displayName") and lh.get("id")
        }
        print(f"  ✓ Found {len(lakehouse_map)} lakehouses")
        return lakehouse_map
    except Exception as e:
        print(f"  ✗ Failed to fetch lakehouses: {e}")
        raise


def fetch_notebooks(workspace_id: str) -> Dict[str, str]:
    """
    Fetch all notebooks in workspace and create name->ID mapping.

    Args:
        workspace_id: Fabric workspace ID

    Returns:
        Dictionary mapping notebook names to IDs
    """
    print("  ℹ️  Fetching notebooks...")
    fabric_client = FabricRestClient()
    url = f"/v1/workspaces/{workspace_id}/notebooks"

    try:
        response = fabric_client.get(url)
        response.raise_for_status()
        notebooks = response.json().get("value", [])
        notebook_map = {
            nb.get("displayName"): nb.get("id")
            for nb in notebooks
            if nb.get("displayName") and nb.get("id")
        }
        print(f"  ✓ Found {len(notebook_map)} notebooks")

        # List the notebooks found for debugging
        if notebook_map:
            print(f"  ℹ️  Notebooks in workspace:")
            for name in sorted(notebook_map.keys()):
                print(f"      • {name}")

        return notebook_map
    except Exception as e:
        print(f"  ✗ Failed to fetch notebooks: {e}")
        raise


def detect_company_prefix(notebook_names: List[str], TECHNICAL_PREFIX: str) -> str:
    """
    Auto-detect company prefix from notebook names.

    Args:
        notebook_names: List of notebook names
        TECHNICAL_PREFIX: The technical prefix (e.g., 'msft')

    Returns:
        Detected company prefix or empty string
    """
    pattern = rf"^(\w+)_{re.escape(TECHNICAL_PREFIX)}_"
    for name in notebook_names:
        match = re.match(pattern, name)
        if match:
            company = match.group(1)
            print(f"  ✓ Auto-detected COMPANY_PREFIX: '{company}'")
            return company
    print(
        f"  ℹ️  No company prefix detected (notebooks use '{TECHNICAL_PREFIX}_*' pattern)"
    )
    return ""


print("✓ Resource discovery functions defined")

In [ ]:
def resolve_placeholder_lakehouse_ids(lakehouse_map: Dict[str, str]) -> Dict[str, str]:
    """
    Resolve lakehouse placeholder names to actual IDs using manifest-based configuration.

    Uses LAKEHOUSE_PLACEHOLDER_CONFIG to map template placeholders to actual lakehouse names.
    For each placeholder:
      1. If manifest_key exists, try build_artifact_name(manifest_key) first
      2. Try each fallback_pattern in order
      3. Try prefixed versions of fallback patterns using build_artifact_name()
      4. Report as missing if none match

    Args:
        lakehouse_map: Dictionary mapping lakehouse names to IDs (from Fabric API)

    Returns:
        Dictionary mapping placeholders to lakehouse IDs
    """
    print("  ℹ️  Resolving lakehouse IDs using manifest configuration...")

    resolved = {}
    missing = []

    for placeholder, config in LAKEHOUSE_PLACEHOLDER_CONFIG.items():
        manifest_key = config["manifest_key"]
        fallback_patterns = config["fallback_patterns"]

        found = False
        matched_name = None

        # Build list of patterns to try, in order of preference
        patterns_to_try = []

        # 1. If manifest_key exists, try prefixed manifest key first (highest priority)
        if manifest_key:
            # Handle hyphens in manifest keys (normalize to underscores)
            normalized_manifest_key = manifest_key.replace("-", "_")
            patterns_to_try.append(build_artifact_name(normalized_manifest_key))

        # 2. Try fallback patterns with prefixes
        for pattern in fallback_patterns:
            patterns_to_try.append(build_artifact_name(pattern))

        # 3. Try original manifest key without prefix (for backward compatibility)
        if manifest_key:
            normalized_manifest_key = manifest_key.replace("-", "_")
            patterns_to_try.append(normalized_manifest_key)

        # 4. Try unprefixed fallback patterns (last resort)
        patterns_to_try.extend(fallback_patterns)

        # Remove duplicates while preserving order
        seen = set()
        patterns_to_try = [p for p in patterns_to_try if not (p in seen or seen.add(p))]

        # Try each pattern until we find a match
        for pattern in patterns_to_try:
            if pattern in lakehouse_map:
                resolved[placeholder] = lakehouse_map[pattern]
                matched_name = pattern
                found = True
                break

        # Report result
        if found:
            manifest_note = (
                f" (from manifest: '{manifest_key}')" if manifest_key else " (external)"
            )
            print(f"    ✓ {placeholder} → {matched_name}{manifest_note}")
        else:
            missing.append(placeholder)
            patterns_tried = ", ".join(patterns_to_try[:3])  # Show first 3 patterns
            more = (
                f" +{len(patterns_to_try)-3} more" if len(patterns_to_try) > 3 else ""
            )
            print(f"    ⚠️  {placeholder} not found (tried: {patterns_tried}{more})")

    # Summary
    if missing:
        print(f"\n  ⚠️  {len(missing)} lakehouse(s) could not be resolved:")
        for placeholder in missing:
            config = LAKEHOUSE_PLACEHOLDER_CONFIG[placeholder]
            if config["manifest_key"]:
                print(
                    f"      • {placeholder} (manifest key: '{config['manifest_key']}')"
                )
            else:
                print(f"      • {placeholder} (external - not in manifest)")
    else:
        print(f"\n  ✓ All {len(resolved)} lakehouses resolved successfully")

    return resolved


print("✓ resolve_placeholder_lakehouse_ids() defined (manifest-based)")

In [ ]:
def resolve_notebook_ids(notebook_map: Dict[str, str]) -> Dict[str, str]:
    """
    Resolve notebook placeholder names to actual IDs.
    
    Args:
        notebook_map: Dictionary mapping notebook names to IDs
        
    Returns:
        Dictionary mapping placeholders to notebook IDs
    """
    print("  ℹ️  Resolving notebook IDs...")

    # Detect the source filename prefix from NOTEBOOK_LAKEHOUSE_MAPPING keys
    # Most notebook filenames in HDS start with a common prefix (e.g., "msft_")
    source_prefix = ""
    if NOTEBOOK_LAKEHOUSE_MAPPING:
        # Get first notebook filename and extract prefix before first underscore after any common pattern
        sample_key = next(iter(NOTEBOOK_LAKEHOUSE_MAPPING.keys()))
        # Check if filenames have a common prefix pattern like "msft_", "custom_", etc.
        if "_" in sample_key.replace(".ipynb", ""):
            parts = sample_key.replace(".ipynb", "").split("_", 1)
            # Check if this prefix is common across multiple notebooks
            potential_prefix = parts[0] + "_"
            matching_count = sum(1 for k in NOTEBOOK_LAKEHOUSE_MAPPING.keys() if k.startswith(potential_prefix))
            if matching_count > len(NOTEBOOK_LAKEHOUSE_MAPPING) * 0.5:  # More than 50% match
                source_prefix = potential_prefix
                print(f"  ℹ️  Detected source filename prefix: '{source_prefix}'")
    
    notebook_placeholders = [
        "fhir_ndjson_bronze_ingestion",
        "bronze_silver_flatten",
        "claims_extract_bronze_ingestion",
        "claims_fhir_conversion",
        "imaging_dicom_extract_bronze_ingestion",
        "imaging_dicom_patch_file_bronze_ingestion",
        "imaging_dicom_silver_metastore_transformation",
        "imaging_dicom_fhir_conversion",
        "omop_silver_gold_transformation",
        "sdoh_raw_extract_bronze_ingestion",
        "sdoh_bronze_silver_flatten",
        "ci_silver_customerinsights_transformation",
        "poa_bronze_silver_transformation",
        "poa_silver_gold_tranformation",
        "cma_silver_gold_transformation",
        "dax_bronze_ingestion",
        "dax_silver_ingestion"
    ]
    
    resolved = {}
    missing = []
    
    for base_name in notebook_placeholders:
        placeholder = f"{base_name}_notebook_id"
        # Construct source filename using detected prefix
        source_filename = f"{source_prefix}{base_name}.ipynb"
        # Use smart prefix logic for notebooks (same as deployment)
        expected_name = build_notebook_display_name(source_filename)
        
        if expected_name in notebook_map:
            resolved[placeholder] = notebook_map[expected_name]
            print(f"    ✓ {placeholder} → {expected_name}")
        else:
            missing.append(placeholder)
            print(f"    ⚠️  {placeholder} not found (expected: {expected_name})")
    
    if missing:
        print(f"  ⚠️  {len(missing)} notebook(s) could not be resolved")
        print(f"\n  💡 ACTION REQUIRED:")
        print(f"      These {len(missing)} notebooks don't exist in the workspace yet.")
        print(f"      To resolve this:")
        print(f"      1. Run 'notebook_deployer.ipynb' to deploy all notebooks")
        print(f"      2. Then re-run this config updater")
        print(f"      3. All {len(notebook_placeholders)} notebooks will be resolved")
    else:
        print(f"  ✓ All {len(resolved)} notebooks resolved")
    
    return resolved

print("✓ resolve_placeholder_lakehouse_ids() defined")

In [ ]:
def copy_configurations_to_admin_lakehouse(lakehouse_map: Dict[str, str]) -> None:
    """
    Copy configuration folders from distribution path to admin lakehouse.
    
    Args:
        lakehouse_map: Dictionary mapping lakehouse names to IDs (from workspace)
    
    Copies:
        1. system-configurations → {admin_lakehouse}/Files/system-configurations
        2. _internal → {admin_lakehouse}/Files/HealthDataManager/DMHConfiguration/_internal
    """
    print("=" * 80)
    print("COPYING CONFIGURATIONS TO ADMIN LAKEHOUSE")
    print("=" * 80)
    
    # Build admin lakehouse name using same pattern as lakehouse deployment
    # Uses manifest key "admin" + dynamic prefixes from common_deployment_config
    admin_lakehouse_name = build_artifact_name("admin")
    print(f"\nℹ️  Target Lakehouse: {admin_lakehouse_name}")
    
    # Validate that admin lakehouse exists in workspace
    if admin_lakehouse_name not in lakehouse_map:
        raise RuntimeError(
            f"Admin lakehouse '{admin_lakehouse_name}' not found in workspace. "
            f"Available lakehouses: {', '.join(sorted(lakehouse_map.keys()))}. "
            f"Please run lakehouses_and_tables_deployer.ipynb first to create the admin lakehouse."
        )
    
    print(f"✓ Admin lakehouse found in workspace (ID: {lakehouse_map[admin_lakehouse_name]})")
    
    # Source base path for configurations
    config_source_base = f"{BASE_DIST_PATH}/healthcare-configuration/{ARTIFACT_VERSION}"
    
    # Target base path in admin lakehouse
    admin_lakehouse_base = f"abfss://{WORKSPACE_NAME}@{ENDPOINT_URI}/{admin_lakehouse_name}.Lakehouse/Files"
    
    print(f"ℹ️  Source: {config_source_base}")
    print(f"ℹ️  Target: {admin_lakehouse_base}\n")
    
    # Define copy operations
    copy_operations = [
        {
            "source": f"{config_source_base}/system-configurations",
            "target": f"{admin_lakehouse_base}/system-configurations",
            "name": "system-configurations"
        },
        {
            "source": f"{config_source_base}/_internal",
            "target": f"{admin_lakehouse_base}/HealthDataManager/DMHConfiguration/_internal",
            "name": "_internal"
        }
    ]
    
    success_count = 0
    error_count = 0
    
    for operation in copy_operations:
        source_path = operation["source"]
        target_path = operation["target"]
        folder_name = operation["name"]
        
        print(f"{'='*60}")
        print(f"📂 Copying: {folder_name}")
        print(f"{'='*60}")
        print(f"  Source: {source_path}")
        print(f"  Target: {target_path}")
        
        try:
            # Check if source exists
            if not mssparkutils.fs.exists(source_path):
                print(f"  ⚠️  Source folder does not exist: {source_path}")
                error_count += 1
                continue
            
            # Create target directory if it doesn't exist
            target_parent = "/".join(target_path.split("/")[:-1])
            if not mssparkutils.fs.exists(target_parent):
                mssparkutils.fs.mkdirs(target_parent)
                print(f"  ✓ Created parent directory")
            
            # Copy the folder using mssparkutils
            mssparkutils.fs.cp(source_path, target_path, recurse=True)
            
            print(f"  ✓ Successfully copied {folder_name}")
            success_count += 1
            
        except Exception as e:
            print(f"  ✗ Failed to copy {folder_name}: {e}")
            error_count += 1
        
        print()
    
    # Summary
    print("=" * 80)
    print("COPY SUMMARY")
    print("=" * 80)
    print(f"  ✓ Successful: {success_count}")
    print(f"  ✗ Failed: {error_count}")
    print(f"  Total: {len(copy_operations)}")
    print("=" * 80)
    
    if error_count > 0:
        raise RuntimeError(f"Failed to copy {error_count} folder(s). Check logs above for details.")


def read_template(template_path: str) -> str:
    """
    Read template configuration file.
    
    Args:
        template_path: Full path to template file
        
    Returns:
        Template file content as string
    """
    print(f"  ℹ️  Reading template file...")
    try:
        content = mssparkutils.fs.head(template_path, 1024 * 1024)
        print(f"  ✓ Template loaded ({len(content):,} bytes)")
        return content
    except Exception as e:
        print(f"  ✗ Failed to read template: {e}")
        raise


def apply_placeholder_replacements(
    content: str,
    workspace_id: str,
    lakehouse_ids: Dict[str, str],
    notebook_ids: Dict[str, str],
    onelake_host: str
) -> str:
    """
    Replace placeholders in template content with actual values.
    
    Args:
        content: Template file content
        workspace_id: Workspace ID
        lakehouse_ids: Resolved lakehouse IDs
        notebook_ids: Resolved notebook IDs
        onelake_host: OneLake host endpoint
        
    Returns:
        Updated content with placeholders replaced
    """
    print("  ℹ️  Applying placeholder replacements...")
    
    # Build full prefix for legacy templates (if needed)
    full_prefix = build_artifact_name("").rstrip("_") if COMPANY_PREFIX or TECHNICAL_PREFIX else ""
    
    replacements = {
        "workspace_id": workspace_id,
        "notebook_prefix": full_prefix,  # For backward compatibility with old templates
        "endpoint_uri": onelake_host,
        **lakehouse_ids,
        **notebook_ids
    }
    
    updated_content = content
    replaced_count = 0
    
    for placeholder, value in replacements.items():
        pattern = f"%%{placeholder}%%"
        if pattern in updated_content:
            updated_content = updated_content.replace(pattern, value)
            replaced_count += 1
    
    print(f"  ✓ Replaced {replaced_count} placeholders")
    return updated_content


def write_output(output_path: str, content: str) -> None:
    """
    Validate and write updated configuration file.
    
    Args:
        output_path: Full path for output file
        content: Content to write
    """
    print("  ℹ️  Validating and writing output...")
    
    # Validate JSON
    try:
        json.loads(content)
        print("  ✓ JSON validation passed")
    except json.JSONDecodeError as e:
        print(f"  ✗ JSON validation failed: {e}")
        raise ValueError(f"Invalid JSON: {e}")
    
    # Write output
    try:
        mssparkutils.fs.put(output_path, content, overwrite=True)
        print(f"  ✓ Configuration written ({len(content):,} bytes)")
    except Exception as e:
        print(f"  ✗ Failed to write output: {e}")
        raise

print("✓ Configuration update functions defined")

---

## Main Orchestration Function

The main function orchestrates the complete configuration update workflow.

In [ ]:
def start_admin_config_updates() -> None:
    """
    Main execution function that orchestrates the complete configuration update workflow.
    Steps:
        1. Display configuration summary
        2. Get workspace context and discover resources
        3. Resolve lakehouse and notebook IDs
        4. Read template file
        5. Apply placeholder replacements
        6. Validate and write output file
        7. Display completion summary
    """
    print("\n" + "=" * 80)
    print("DEPLOYMENT PARAMETER CONFIGURATION UPDATER")
    print("=" * 80)
    # Display configuration summary
    print("\nℹ️  Configuration Summary:")
    print(f"  Workspace: {WORKSPACE_NAME}")
    print(f"  BASE_DIST_PATH: {BASE_DIST_PATH}")
    print(f"\nInput File:")
    print(f"  TEMPLATE_FULL_PATH: {TEMPLATE_FULL_PATH}")
    print(f"\nOutput File:")
    print(f"  OUTPUT_FULL_PATH: {OUTPUT_FULL_PATH}")
    print("=" * 80 + "\n")
    try:
        # Step 1: Get workspace context
        print("=" * 80)
        print("STEP 1: DISCOVER RESOURCES")
        print("=" * 80)
        workspace_id = get_workspace_id()
        lakehouse_map = fetch_lakehouses(workspace_id)
        notebook_map = fetch_notebooks(workspace_id)
        # Display configured prefixes
        print(f"  ℹ️  Using prefixes from common config:")
        print(f"     COMPANY_PREFIX: '{COMPANY_PREFIX}'")
        print(f"     TECHNICAL_PREFIX: '{TECHNICAL_PREFIX}'")
        # Optionally auto-detect company prefix if not configured
        if not COMPANY_PREFIX:
            detected_company = detect_company_prefix(list(notebook_map.keys()), TECHNICAL_PREFIX)
            print(f"  ℹ️  Auto-detection result: '{detected_company}'")
        
        # Step 2: Resolve IDs
        print("\n" + "=" * 80)
        print("STEP 2: RESOLVE RESOURCE IDs")
        print("=" * 80)
        lakehouse_ids = resolve_placeholder_lakehouse_ids(lakehouse_map)
        notebook_ids = resolve_notebook_ids(notebook_map)
        
        # Step 3: Update configuration
        print("\n" + "=" * 80)
        print("STEP 3: UPDATE CONFIGURATION")
        print("=" * 80)
        template_content = read_template(TEMPLATE_FULL_PATH)
        updated_content = apply_placeholder_replacements(
            template_content,
            workspace_id,
            lakehouse_ids,
            notebook_ids,
            ENDPOINT_URI
        )
        write_output(OUTPUT_FULL_PATH, updated_content)
        # Display completion summary
        print("\n" + "=" * 80)
        print("✅ CONFIGURATION UPDATE COMPLETE")
        print("=" * 80)
        print(f"\nSummary:")
        print(f"  • Workspace ID: {workspace_id}")
        print(f"  • Company Prefix: '{COMPANY_PREFIX}'")
        print(f"  • Technical Prefix: '{TECHNICAL_PREFIX}'")
        print(f"  • Lakehouses Resolved: {len(lakehouse_ids)}")
        print(f"  • Notebooks Resolved: {len(notebook_ids)}")
        print(f"  • Output File: {OUTPUT_FULL_PATH}")
        print(f"\nNext Steps:")
        print(f"  1. Verify output file contains correct resource IDs")
        print(f"  2. Review any unresolved placeholders")
        print(f"  3. Use updated configuration in deployment pipelines")
        print("=" * 80)
    except Exception as e:
        print("\n" + "=" * 80)
        print("✗ ERROR")
        print("=" * 80)
        print(f"  {str(e)}")
        print("=" * 80)
        raise


def start_admin_config_updates_with_copy() -> None:
    """
    Extended configuration update workflow that includes copying configuration folders.
    
    Steps:
        1. Display configuration summary
        2. Get workspace context and discover resources
        3. Resolve lakehouse and notebook IDs
        4. Update configuration file (generate deploymentParametersConfiguration.json)
        5. Copy configuration folders to admin lakehouse
        6. Display completion summary
    """
    print("\n" + "=" * 80)
    print("DEPLOYMENT PARAMETER CONFIGURATION UPDATER (WITH COPY)")
    print("=" * 80)
    # Display configuration summary
    print("\nℹ️  Configuration Summary:")
    print(f"  Workspace: {WORKSPACE_NAME}")
    print(f"  BASE_DIST_PATH: {BASE_DIST_PATH}")
    print(f"\nInput File:")
    print(f"  TEMPLATE_FULL_PATH: {TEMPLATE_FULL_PATH}")
    print(f"\nOutput File:")
    print(f"  OUTPUT_FULL_PATH: {OUTPUT_FULL_PATH}")
    print("=" * 80 + "\n")
    try:
        # Step 1: Get workspace context
        print("=" * 80)
        print("STEP 1: DISCOVER RESOURCES")
        print("=" * 80)
        workspace_id = get_workspace_id()
        lakehouse_map = fetch_lakehouses(workspace_id)
        notebook_map = fetch_notebooks(workspace_id)
        # Display configured prefixes
        print(f"  ℹ️  Using prefixes from common config:")
        print(f"     COMPANY_PREFIX: '{COMPANY_PREFIX}'")
        print(f"     TECHNICAL_PREFIX: '{TECHNICAL_PREFIX}'")
        # Optionally auto-detect company prefix if not configured
        if not COMPANY_PREFIX:
            detected_company = detect_company_prefix(list(notebook_map.keys()), TECHNICAL_PREFIX)
            print(f"  ℹ️  Auto-detection result: '{detected_company}'")
        
        # Step 2: Resolve IDs
        print("\n" + "=" * 80)
        print("STEP 2: RESOLVE RESOURCE IDs")
        print("=" * 80)
        lakehouse_ids = resolve_placeholder_lakehouse_ids(lakehouse_map)
        notebook_ids = resolve_notebook_ids(notebook_map)
        
        # Step 3: Update configuration (MUST happen before copy)
        print("\n" + "=" * 80)
        print("STEP 3: UPDATE CONFIGURATION FILE")
        print("=" * 80)
        template_content = read_template(TEMPLATE_FULL_PATH)
        updated_content = apply_placeholder_replacements(
            template_content,
            workspace_id,
            lakehouse_ids,
            notebook_ids,
            ENDPOINT_URI
        )
        write_output(OUTPUT_FULL_PATH, updated_content)
        
        # Step 4: Copy configurations to admin lakehouse
        print("\n" + "=" * 80)
        print("STEP 4: COPY CONFIGURATIONS TO ADMIN LAKEHOUSE")
        print("=" * 80)
        copy_configurations_to_admin_lakehouse(lakehouse_map)
        
        # Display completion summary
        print("\n" + "=" * 80)
        print("✅ CONFIGURATION UPDATE COMPLETE (WITH COPY)")
        print("=" * 80)
        print(f"\nSummary:")
        print(f"  • Workspace ID: {workspace_id}")
        print(f"  • Company Prefix: '{COMPANY_PREFIX}'")
        print(f"  • Technical Prefix: '{TECHNICAL_PREFIX}'")
        print(f"  • Lakehouses Resolved: {len(lakehouse_ids)}")
        print(f"  • Notebooks Resolved: {len(notebook_ids)}")
        print(f"  • Configuration File: {OUTPUT_FULL_PATH}")
        print(f"  • Configurations Copied: system-configurations, _internal")
        print(f"\nNext Steps:")
        print(f"  1. Verify output file contains correct resource IDs")
        print(f"  2. Verify configurations copied to admin lakehouse")
        print(f"  3. Review any unresolved placeholders")
        print(f"  4. Use updated configuration in deployment pipelines")
        print("=" * 80)
    except Exception as e:
        print("\n" + "=" * 80)
        print("✗ ERROR")
        print("=" * 80)
        print(f"  {str(e)}")
        print("=" * 80)
        raise

print("✓ start_admin_config_updates() defined")
print("✓ start_admin_config_updates_with_copy() defined")

---

## Execute Configuration Update

Run the main function to execute the configuration update workflow.

**⚠️ Important:**

1. In `common_deployment_config`, make sure `WORKSPACE_NAME` (and related settings like `ARTIFACT_LAKEHOUSE_NAME`) are correct and case-sensitive.
2. In this notebook, verify the derived `TEMPLATE_FULL_PATH` and `OUTPUT_FULL_PATH` values printed in the configuration cell.
3. Choose one of the execution options below:

### Option 1: Configuration Update Only (Original)
Updates configuration file with resource IDs only.
```python
start_admin_config_updates()
```

### Option 2: Configuration Update + Copy (New Feature)
Updates configuration file FIRST with resource IDs, THEN copies configuration folders to admin lakehouse:
1. Generate `deploymentParametersConfiguration.json` with resolved resource IDs
2. Copy folders:
   - `system-configurations` → `{admin_lakehouse}/Files/system-configurations`
   - `_internal` → `{admin_lakehouse}/Files/HealthDataManager/DMHConfiguration/_internal`

**Prerequisites**: Admin lakehouse must be created (via `lakehouses_and_tables_deployer.ipynb`).
```python
start_admin_config_updates_with_copy()
```

In [ ]:
# Execute the configuration update (choose one option):

# Option 1: Configuration update only (original functionality)
# start_admin_config_updates()

# Option 2: Configuration update + copy folders to admin lakehouse (new feature)
start_admin_config_updates_with_copy()